# Phishing website classification: training and development

This notebook preserves the original comparison of logistic regression, KNN,
a decision tree, and a random forest. The selected model uses 250 trees,
maximum depth 22, balanced bootstrap samples, and random seed 42.

The data contains website features, not email text. The original reported F1
uses **1 = legitimate** as the positive class. **-1 = phishing** is the other
class. The companion training script reports both class-specific F1 scores.

The 30-feature forest is exported before the later two-component PCA experiments
replace the model variables. PCA is for visualization, not the deployed model.
The webpage uses a separately trained forest on up to 22 collectable features.
When evidence is missing, it fits the original training rows using exactly the
available columns. See the README and FEATURES.md. Missing website features are never filled with invented values.

Personal identifiers and cached outputs have been removed. Comments have been
revised. The original split and selected hyperparameters are retained. These
results measure performance on this historical feature dataset, not detection
accuracy for present-day emails. Setup: install `requirements-notebook.txt`.


In [ ]:
# Locate the project when the notebook starts in the root or notebooks folder.
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'phishing_websites.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# The supplied dataset has already been converted from ARFF to CSV.



## Load and split the data

In [ ]:
# Load the dataset and separate its 30 input features from the target labels.
# Result = -1 means phishing; Result = 1 means legitimate.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split


df = pd.read_csv(DATA_PATH)
X = df.drop(columns='Result')
Y = df['Result']
print(X.shape)
print(Y.shape)


In [ ]:
# Count each class before creating the training and test split.
Y.value_counts()


In [ ]:
# Keep 70% for training and 30% for testing, using the original fixed seed.
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.7, random_state=42)


## Evaluation helpers

In [ ]:
# Print classification metrics consistently across the experiments.
# These functions retain the original positive class of 1 (legitimate).
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def print_metrics(Y_t, Y_p):
    print(f"accuracy: {accuracy_score(Y_t, Y_p): .3f}")
    print(f"precision: {precision_score(Y_t, Y_p): .3f}")
    print(f"recall: {recall_score(Y_t, Y_p): .3f}")
    print()

def f1_score_print(Y_t, Y_p):
    print(f"F1 score: {f1_score(Y_t, Y_p): .3f}")
    print()


## Logistic regression

In [ ]:
# Tune logistic regression using ten-fold cross-validation on the training set.
# C controls the inverse strength of L2 regularization.
# The original search metric is F1 for the legitimate class.
from sklearn.model_selection import cross_val_score

l2_inverse_vals = np.linspace(1.0, 0.001, 100)
validation_performance = []
for param in l2_inverse_vals:

    LR = LogisticRegression(C = param, class_weight='balanced', random_state=42)

    scores = cross_val_score(LR, X_train, Y_train, cv=10, scoring='f1')
    validation_performance.append(scores.mean())
print(f"the optimal l2 value is: {l2_inverse_vals[np.argmax(validation_performance)]}\n")


In [ ]:
# Plot cross-validation performance across the tested regularization settings.
plt.plot(l2_inverse_vals, validation_performance)
tick_positions = np.linspace(1.0, 0.001, 10)
plt.xticks(tick_positions)
plt.xlabel('Inverse of L2 Regularization amount')
plt.ylabel('Cross-Validation F1 score')
plt.title('Logistic regression performance as L2 regularization increases')
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Compare default logistic regression with the selected C value of 0.072.
optimized_l2 = 0.072

LR_orig = LogisticRegression(class_weight='balanced', random_state=42)
LR_orig.fit(X_train, Y_train)
Pred_test_orig = LR_orig.predict(X_test)

LR_new = LogisticRegression(C = optimized_l2, class_weight='balanced', random_state=42)
LR_new.fit(X_train, Y_train)
Pred_test_new = LR_new.predict(X_test)

print("the testing F1 performance for LC original is:")
f1_score_print(Y_test, Pred_test_orig)
print("the testing F1 performance for optimized LC is:")
f1_score_print(Y_test, Pred_test_new)


## K-nearest neighbours

In [ ]:
# Tune the number of neighbours using the original candidate list.
# Convert each candidate to an integer before fitting KNN.
K_vals = np.linspace(1, 21, 10)
validation_performance = []
for K in K_vals:

    KNN = KNeighborsClassifier(n_neighbors=int(K))

    scores = cross_val_score(KNN, X_train, Y_train, cv=10, scoring='f1')
    validation_performance.append(scores.mean())
print(f"the optimal K value is: {int(K_vals[np.argmax(validation_performance)])}\n")


In [ ]:
# Plot KNN performance for the tested neighbour counts.
plt.plot(K_vals, validation_performance)
plt.xticks(K_vals.astype(int))
plt.xlabel('Number of neighbours considered')
plt.ylabel('Cross-Validation F1 score')
plt.title('K-Nearest Neighbours performance as number of neighbours increases')
plt.tight_layout()
plt.show()


In [ ]:
# Compare default KNN with the selected one-neighbour model.
optimized_num_neighbours = 1

KNN_orig = KNeighborsClassifier()
KNN_orig.fit(X_train, Y_train)
Pred_test_orig = KNN_orig.predict(X_test)

KNN_new = KNeighborsClassifier(n_neighbors=1)
KNN_new.fit(X_train, Y_train)
Pred_test_new = KNN_new.predict(X_test)

print("the testing F1 performance for the default KNN is:")
f1_score_print(Y_test, Pred_test_orig)
print("the testing F1 performance for optimized KNN is:")
f1_score_print(Y_test, Pred_test_new)


## Decision tree

In [ ]:
# Tune the maximum decision-tree depth using ten-fold cross-validation.
# Tree depth does not specify how many distinct features the tree uses.
depth_vals = np.linspace(1, 61, 60)
validation_performance = []
for D in depth_vals:

    DT = DecisionTreeClassifier(max_depth=int(D), class_weight='balanced', random_state=42)

    scores = cross_val_score(DT, X_train, Y_train, cv=10, scoring='f1')
    validation_performance.append(scores.mean())
print(f"the optimal depth value is: {int(depth_vals[np.argmax(validation_performance)])}\n")


In [ ]:
# Plot decision-tree performance with a smaller set of axis labels.
plt.plot(depth_vals, validation_performance)

xtick_vals = np.linspace(1, 61, 10)

plt.xticks(xtick_vals.astype(int))
plt.xlabel('Max allowed depth of tree')
plt.ylabel('Cross-Validation F1 score')
plt.title('Decision Tree Classifier performance as max tree depth increases')
plt.tight_layout()
plt.show()


In [ ]:
# Compare the default decision tree with a maximum depth of 24.
# The fitted tree may stop before reaching its allowed maximum depth.
optimized_max_depth = 24

DT_orig = DecisionTreeClassifier(class_weight='balanced', random_state=42)
DT_orig.fit(X_train, Y_train)
Pred_test_orig = DT_orig.predict(X_test)

DT_new = DecisionTreeClassifier(max_depth= optimized_max_depth, class_weight='balanced', random_state=42)
DT_new.fit(X_train, Y_train)
Pred_test_new = DT_new.predict(X_test)

print("the testing F1 performance for the default decision tree classifier is:")
f1_score_print(Y_test, Pred_test_orig)
print("the testing F1 performance for the optimized decision tree classifier is:")
f1_score_print(Y_test, Pred_test_new)

print(f"the depth of the original tree is: {DT_orig.get_depth()}")
print(f"the depth of the optimized tree is: {DT_new.get_depth()}")


## Random forest

In [ ]:
# Tune random-forest tree count and maximum depth.
# Balance the classes separately within each bootstrap sample.
depth_vals = np.linspace(1, 22,5)

depth_vals = depth_vals.astype(int)
num_estimators_vals = [50, 100, 150, 200, 250, 300]
validation_performance = []
saved_params = []
for N in num_estimators_vals:
    for D in depth_vals:

        RF = RandomForestClassifier(n_estimators=N,max_depth=D, class_weight='balanced_subsample', random_state=42)

        scores = cross_val_score(RF, X_train, Y_train, cv=10, scoring='f1')
        validation_performance.append(scores.mean())
        saved_params.append([N,D])
print(f"the optimal number of estimators was: {saved_params[np.argmax(validation_performance)][0]}\n")
print(f"the optimal depth value is: {saved_params[np.argmax(validation_performance)][1]}\n")


In [ ]:
# Plot the random-forest parameter combinations and validation scores.
x_positions = range(len(saved_params))
plt.plot(x_positions, validation_performance)
param_strings = [f"({int(N)}, {int(D)})" for N,D in saved_params]
plt.xticks(x_positions, param_strings, rotation=90)
plt.xlabel('[num estimators, max depth]')
plt.ylabel('Cross-Validation F1 score')
plt.title('Random Forest Classifier performance as the number of trees and tree depth increases')
plt.tight_layout()
plt.show()


In [ ]:
# Compare the default random forest with 250 trees and maximum depth 22.
optimized_max_depth = 22
optimized_num_estimators = 250

RF_orig = RandomForestClassifier(class_weight='balanced_subsample', random_state=42)
RF_orig.fit(X_train, Y_train)
Pred_test_orig = RF_orig.predict(X_test)

RF_new = RandomForestClassifier(n_estimators=optimized_num_estimators,max_depth=optimized_max_depth, class_weight='balanced_subsample', random_state=42)
RF_new.fit(X_train, Y_train)
Pred_test_new = RF_new.predict(X_test)

print("the testing F1 performance for the default random forest classifier is:")
f1_score_print(Y_test, Pred_test_orig)
print("the testing F1 performance for the optimized random forest classifier is:")
f1_score_print(Y_test, Pred_test_new)


## Compare the selected models

In [ ]:
# Refit all four selected models on the original 30 features.
# Keep these results separate from the later PCA visualization models.
LR = LogisticRegression(C = optimized_l2, class_weight='balanced', random_state=42)
LR.fit(X_train, Y_train)
Pred_test_LR = LR.predict(X_test)

KNN = KNeighborsClassifier(n_neighbors=1)
KNN.fit(X_train, Y_train)
Pred_test_KNN = KNN.predict(X_test)

DT = DecisionTreeClassifier(max_depth= 24, class_weight='balanced', random_state=42)
DT.fit(X_train, Y_train)
Pred_test_DT = DT.predict(X_test)

RF = RandomForestClassifier(n_estimators=250,max_depth=22, class_weight='balanced_subsample', random_state=42)
RF.fit(X_train, Y_train)
Pred_test_RF = RF.predict(X_test)

model_scores = [f1_score(Y_test, Pred_test_LR), f1_score(Y_test, Pred_test_KNN), f1_score(Y_test, Pred_test_DT), f1_score(Y_test, Pred_test_RF)]
models_tested = ["LR", "KNN", "DT", "RF"]


In [ ]:
# Compare held-out F1 scores for the legitimate class.
cmap= plt.get_cmap('hsv')
colors = cmap(np.linspace(0,0.7, 4))
plt.bar(models_tested, model_scores, color = colors)
plt.title('Performance comparison for various models on classifying phishing and safe URLs')
plt.xlabel('Models tested')
plt.ylabel('F1 score')
plt.show()


In [ ]:
# Report the model scores and the difference in percentage points.
print(models_tested)
print(model_scores)
print(f"the difference between the best and worst performing model is {(model_scores[3] - model_scores[0])*100 :.3f}%")


## Export the selected full-feature model
This saves the fitted 30-feature forest before PCA replaces the model variables.

In [ ]:
# Save the selected forest and the exact feature order it was trained on.
from joblib import dump
full_model = RF
dump(full_model, MODEL_DIR / 'website_random_forest.joblib', compress=3)
print('Saved the full website model with', full_model.n_features_in_, 'features.')


## PCA visualization experiments

In [ ]:
# Fit PCA on the training data and calculate cumulative explained variance.
pca = PCA(random_state=42)
pca.fit(X_train)

pca_ratios = list(pca.explained_variance_ratio_)

pca_ratios.insert(0,0.0)
pca_ratios_cumsum = np.cumsum(pca_ratios)
principal_components = list(range(0,31,1))
print(principal_components)
print(pca_ratios_cumsum)


In [ ]:
# Show how much variance is retained as more components are included.
plt.plot(principal_components, pca_ratios_cumsum)

plt.xticks(principal_components)
plt.axvline(x=2, color='grey', linestyle='--', linewidth=1.5)
plt.xlabel('Principal component number')
plt.ylabel('cumulative variance for principal components')
plt.title('Number of principal components against the cumulative variance they explain')
plt.tight_layout()
plt.show()


In [ ]:
# Fit a two-component PCA projection for visualization only.
pca = PCA(n_components=2, random_state=42)
pca.fit(X_train)
print(pca.explained_variance_ratio_)

plottable_test_data = pca.transform(X_test)

print(plottable_test_data.shape)


In [ ]:
# Display the test data in the two-dimensional PCA projection.
plt.scatter(plottable_test_data[:,0], plottable_test_data[:,1], c=Y_test, cmap='viridis', s = 3)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Plot of dimensionally reduced URL phishing test data')
plt.colorbar(label = 'Phishing vs real')
plt.show()


In [ ]:
# Fit separate visualization models on the two PCA components.
# These are not the 30-feature models used for the primary results.
# Build a grid of points to display their decision boundaries.
plottable_train_data = pca.transform(X_train)

LR = LogisticRegression(C = optimized_l2, class_weight='balanced', random_state=42)
LR.fit(plottable_train_data, Y_train)

KNN = KNeighborsClassifier(n_neighbors=1)
KNN.fit(plottable_train_data, Y_train)

DT = DecisionTreeClassifier(max_depth= 24, class_weight='balanced', random_state=42)
DT.fit(plottable_train_data, Y_train)

RF = RandomForestClassifier(n_estimators=250,max_depth=22, class_weight='balanced_subsample', random_state=42)
RF.fit(plottable_train_data, Y_train)

xmesh, ymesh = np.meshgrid(np.linspace(-6.5,2.0,300), np.linspace(-4.5,4.5,300))

mesh_points = np.c_[xmesh.ravel(),ymesh.ravel()]


In [ ]:
# Plot the logistic-regression decision boundary in the PCA projection.
mesh_preds = LR.predict(mesh_points)
print(mesh_preds.shape)
mesh_preds = mesh_preds.reshape(xmesh.shape)
print(mesh_preds.shape)

plt.contourf(xmesh, ymesh, mesh_preds, alpha = 0.3)
plt.scatter(plottable_test_data[:,0], plottable_test_data[:,1], c=Y_test, cmap='viridis', s = 3)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Plot of PCA reduced URL phishing test data classified by LR')
plt.colorbar(label = 'Phishing vs real')
plt.show()


In [ ]:
# Plot the KNN decision boundary in the PCA projection.
mesh_preds = KNN.predict(mesh_points)
print(mesh_preds.shape)
mesh_preds = mesh_preds.reshape(xmesh.shape)
print(mesh_preds.shape)

plt.contourf(xmesh, ymesh, mesh_preds, alpha = 0.3)
plt.scatter(plottable_test_data[:,0], plottable_test_data[:,1], c=Y_test, cmap='viridis', s = 3)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Plot of PCA reduced URL phishing test data classified by KNN')
plt.colorbar(label = 'Phishing vs real')
plt.show()


In [ ]:
# Plot the decision-tree decision boundary in the PCA projection.
mesh_preds = DT.predict(mesh_points)
print(mesh_preds.shape)
mesh_preds = mesh_preds.reshape(xmesh.shape)
print(mesh_preds.shape)

plt.contourf(xmesh, ymesh, mesh_preds, alpha = 0.3)
plt.scatter(plottable_test_data[:,0], plottable_test_data[:,1], c=Y_test, cmap='viridis', s = 3)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Plot of PCA reduced URL phishing test data classified by DT')
plt.colorbar(label = 'Phishing vs real')
plt.show()


In [ ]:
# Plot the random-forest decision boundary in the PCA projection.
mesh_preds = RF.predict(mesh_points)
print(mesh_preds.shape)
mesh_preds = mesh_preds.reshape(xmesh.shape)
print(mesh_preds.shape)

plt.contourf(xmesh, ymesh, mesh_preds, alpha = 0.3)
plt.scatter(plottable_test_data[:,0], plottable_test_data[:,1], c=Y_test, cmap='viridis', s = 3)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('Plot of PCA reduced URL phishing test data classified by RF')
plt.colorbar(label = 'Phishing vs real')
plt.show()
